# Prosperity 4 Tutorial Round Explorer

This notebook gives a practical first-pass dashboard for `INTARIAN PEPPER ROOTS` and `ASH-COATED OSMIUM`:
- Price dynamics (bid/mid/ask)
- Spread and liquidity behavior
- Order-book imbalance and volatility
- Trade flow, quantity, and notional patterns

In [17]:
from pathlib import Path
import re
import sys
from typing import cast

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parents[1] if Path.cwd().name == 'analysis' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from prosperity4.analysis.data import read_all_round2_prices

prices = read_all_round2_prices()

round2_dir = repo_root / 'data' / 'ROUND_2'
trade_frames: list[pd.DataFrame] = []
for path in sorted(round2_dir.glob('trades_round_2_day_*.csv')):
    day_match = re.search(r'day_(-?\d+)$', path.stem)
    day = int(day_match.group(1)) if day_match else 0
    frame = cast(pd.DataFrame, pd.read_csv(filepath_or_buffer=str(path), sep=';'))
    frame['day'] = day
    trade_frames.append(frame)

trades = pd.concat(trade_frames, ignore_index=True) if trade_frames else pd.DataFrame()

print(f'Prices shape: {prices.shape}')
print(f'Trades shape: {trades.shape}')
prices.head()

Prices shape: (60000, 17)
Trades shape: (2391, 8)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,INTARIAN_PEPPER_ROOT,10994.0,9.0,NaN,NaN,NaN,NaN,11009.0,25.0,NaN,NaN,NaN,NaN,11001.5,0.0
1,-1,0,ASH_COATED_OSMIUM,9982.0,21.0,NaN,NaN,NaN,NaN,10000.0,13.0,10003.0,21.0,NaN,NaN,9991.0,0.0
2,-1,100,ASH_COATED_OSMIUM,9984.0,15.0,NaN,NaN,NaN,NaN,10000.0,15.0,10003.0,26.0,NaN,NaN,9992.0,0.0
3,-1,100,INTARIAN_PEPPER_ROOT,10994.0,8.0,10991.0,22.0,NaN,NaN,11006.0,8.0,11009.0,22.0,NaN,NaN,11000.0,0.0
4,-1,200,ASH_COATED_OSMIUM,9985.0,15.0,9982.0,30.0,NaN,NaN,10001.0,15.0,NaN,NaN,NaN,NaN,9993.0,0.0


In [18]:
df = prices.copy()
df = df.sort_values(['product', 'day', 'timestamp']).reset_index(drop=True)

df['time_index'] = df['day'] * 1_000_000 + df['timestamp']
df['spread'] = df['ask_price_1'] - df['bid_price_1']
df['mid'] = (df['ask_price_1'] + df['bid_price_1']) / 2

bid_v = df['bid_volume_1'].abs()
ask_v = df['ask_volume_1'].abs()
top_depth = bid_v + ask_v

df['imbalance'] = (bid_v - ask_v).where(top_depth > 0, 0) / top_depth.where(top_depth > 0, 1)
df['microprice'] = ((df['ask_price_1'] * bid_v) + (df['bid_price_1'] * ask_v)).where(top_depth > 0, df['mid']) / top_depth.where(top_depth > 0, 1)
df['mid_return'] = df.groupby('product')['mid'].pct_change().fillna(0.0)
df['rolling_vol_100'] = (
    df.groupby('product')['mid_return']
    .rolling(100)
    .std()
    .reset_index(level=0, drop=True)
)

trades = trades.rename(columns={'symbol': 'product'})
trades['time_index'] = trades['day'] * 1_000_000 + trades['timestamp']
trades['notional'] = trades['price'] * trades['quantity']

spread_stats = (
    df.groupby('product')['spread']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .sort_index()
)
spread_stats

,count,mean,median,std,min,max
product,,,,,,
ASH_COATED_OSMIUM,27708,16.234156,16.0,2.520303,5.0,22.0
INTARIAN_PEPPER_ROOT,27724,14.121664,14.0,2.788202,2.0,22.0


In [19]:
liquidity_stats = (
    df.groupby('product')
    .agg(
        mean_spread=('spread', 'mean'),
        mean_depth=('bid_volume_1', lambda s: s.abs().mean()),
        mean_ask_depth=('ask_volume_1', lambda s: s.abs().mean()),
        mean_abs_imbalance=('imbalance', lambda s: s.abs().mean()),
    )
    .sort_index()
)
liquidity_stats

,mean_spread,mean_depth,mean_ask_depth,mean_abs_imbalance
product,,,,
ASH_COATED_OSMIUM,16.234156,14.240036,14.202038,0.115411
INTARIAN_PEPPER_ROOT,14.121664,11.588097,11.563327,0.093465


In [20]:
sample = (
    df.groupby('product', as_index=False, group_keys=False)
    .head(3000)
    .copy()
)

price_panel = sample.melt(
    id_vars=['product', 'time_index'],
    value_vars=['bid_price_1', 'mid', 'ask_price_1'],
    var_name='series',
    value_name='price',
)

fig_price = px.line(
    price_panel,
    x='time_index',
    y='price',
    color='series',
    facet_row='product',
    title='Top-of-Book Prices (first 3000 points per product)',
)
fig_price.update_layout(height=700)
fig_price.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
fig_spread = px.line(
    sample,
    x='time_index',
    y='spread',
    color='product',
    title='Bid/Ask Spread Over Time (first 3000 points per product)',
)
fig_spread.show()

In [ ]:
fig_spread_dist = px.box(
    df,
    x='product',
    y='spread',
    color='product',
    points=False,
    title='Spread Distribution by Product',
)
fig_spread_dist.show()

In [ ]:
fig_imbalance = px.line(
    sample,
    x='time_index',
    y='imbalance',
    color='product',
    title='Top-of-Book Imbalance Over Time (first 3000 points per product)',
)
fig_imbalance.show()

In [ ]:
vol_sample = df.groupby('product', as_index=False, group_keys=False).head(4000)
fig_vol = px.line(
    vol_sample,
    x='time_index',
    y='rolling_vol_100',
    color='product',
    title='Rolling Mid-Return Volatility (window=100)',
)
fig_vol.show()

In [ ]:
trade_activity = (
    trades.groupby(['product', 'time_index'], as_index=False)
    .agg(quantity=('quantity', 'sum'), notional=('notional', 'sum'))
)

fig_qty = px.line(
    trade_activity,
    x='time_index',
    y='quantity',
    color='product',
    title='Executed Quantity Over Time',
)
fig_qty.show()

fig_notional = px.line(
    trade_activity,
    x='time_index',
    y='notional',
    color='product',
    title='Executed Notional Over Time',
)
fig_notional.show()

In [ ]:
fig_trade_price = px.histogram(
    trades,
    x='price',
    color='product',
    barmode='overlay',
    nbins=60,
    title='Trade Price Distribution',
)
fig_trade_price.show()

In [ ]:
signal_view = df[['product', 'imbalance', 'mid_return']].copy()
fig_signal = px.scatter(
    signal_view.sample(min(len(signal_view), 8000), random_state=7),
    x='imbalance',
    y='mid_return',
    color='product',
    opacity=0.35,
    title='Imbalance vs Next Mid Return Proxy (sanity-check scatter)',
)
fig_signal.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Generate one figure per Round 1 product for the requested mid/signal view.
products_for_signal = ['ASH_COATED_OSMIUM', 'INTARIAN_PEPPER_ROOT']

for product_for_signal in products_for_signal:
    prices = (
        df.loc[df['product'] == product_for_signal, 'mid']
        .dropna()
        .reset_index(drop=True)
    )

    # Requested signal: rolling z-score (50) smoothed with a 100-tick moving average.
    signal = ((prices - prices.rolling(50).mean()) / prices.rolling(50).std()).rolling(100).mean()

    fig_mid_and_signal = make_subplots(rows=2, cols=1, shared_xaxes=True)
    fig_mid_and_signal.add_trace(
        go.Scatter(x=prices.index, y=prices, name=f'{product_for_signal} Mid price'),
        row=1,
        col=1,
    )
    fig_mid_and_signal.add_trace(
        go.Scatter(
            x=prices.index,
            y=signal,
            name='Signal',
        ),
        row=2,
        col=1,
    )
    fig_mid_and_signal.update_layout(
        height=800,
        title=f'{product_for_signal}: Mid Price and Rolling Z-Score Signal',
    )
    fig_mid_and_signal.show()

